In [1]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader, random_split
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 👇 custom loader يتجاهل الصور البايظة
def safe_loader(path):
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            return img.convert("RGB")
    except:
        return None

class SafeImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = safe_loader(path)
        if sample is None:
            return self.__getitem__((index + 1) % len(self.samples))
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, target

def load_data(data_dir):
    dataset = SafeImageFolder(data_dir, transform=transform)
    return dataset

def split_dataloader(data, train_split):
    train_size = int(train_split * len(data))
    val_size = len(data) - train_size
    train_data, val_data = random_split(data, [train_size, val_size])

    trainL = DataLoader(train_data, batch_size=32, shuffle=True)
    valL = DataLoader(val_data, batch_size=32, shuffle=False)

    return trainL, valL

In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

def build_model(num_classes, device):
    model = models.resnet50(weights="IMAGENET1K_V1")
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    model = model.to(device)
    return model

def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False

    for param in model.fc.parameters():
        param.requires_grad = True

    return model

def get_optimizer(model, lr):
    return torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

def get_loss_function():
    return nn.CrossEntropyLoss()

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct.double() / len(dataloader.dataset)

    return epoch_loss, epoch_acc.item()

In [3]:
# ⚙️ Settings
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 🔴 حطي اسم الداتا الصحيح من Kaggle
data_dir = "/kaggle/input/drone-detection-dataset/BirdVsDroneVsAirplane"

# Load dataset
dataset = load_data(data_dir)
train_loader, val_loader = split_dataloader(dataset, 0.8)

num_classes = len(dataset.classes)
print("Classes:", dataset.classes)

# Build model
model = build_model(num_classes, device)
model = freeze_backbone(model)

optimizer = get_optimizer(model, lr=0.001)
criterion = get_loss_function()

# 🚀 Training (1 epoch تجربة)
epochs = 1

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Train Accuracy:", train_acc)

# 💾 Save model
torch.save(model.state_dict(), "resnet50_model.pth")
print("Model Saved Successfully ✅")

Using device: cuda
Classes: ['Aeroplanes', 'Birds', 'Drones']
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]


Epoch 1
Train Loss: 0.5584781169891357
Train Accuracy: 0.7956089911134344
Model Saved Successfully ✅
